# Notebook 05 — Adaptive RAG & Practical Utility (RQ5)

**Goal:** Show that QPP-guided adaptive retrieval yields consistent generation
improvements over the baseline RAG — especially for Low-QPP (hard) queries.
Also compares against the learning-free query-length heuristic baseline.

Reproduces **Table 11**, **Table 12**, **Figure 6**, and the
length-heuristic comparison from Section 6.

Reference: Sinha & Chakma (2026), Sections 3.7, 4, 5.5, 6

---
**Sections**
1. Environment setup
2. Simulate baseline vs QPP-guided vs heuristic results
3. Table 11 — Overall generation quality comparison
4. Table 12 — Per QPP-group gains
5. Figure 6 — Proof-of-concept example
6. Heuristic comparison (Section 6)
7. Full end-to-end run (optional — requires GPU + datasets)
8. Takeaways

## 1. Environment Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
np.random.seed(42)

print('Setup complete.')

## 2. Simulate Results (Baseline vs QPP-Guided vs Heuristic)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Option A: load real pipeline outputs
# import json
# baseline_rows   = [json.loads(l) for l in open('../outputs/predictions/generation_results.jsonl')]
# qpp_rows        = [json.loads(l) for l in open('../outputs/predictions/adaptive_rag_results.jsonl')]
#
# Option B: synthetic data calibrated to paper Table 11 / Table 12
# ─────────────────────────────────────────────────────────────────────────────

N = 400   # paper uses ~1000 queries for MS MARCO Document

# Ground-truth QPP scores (proxy for query difficulty)
qpp_scores = np.random.beta(2, 3, N)   # most queries are medium-hard

# Tertile bins
t1, t2 = np.percentile(qpp_scores, 33), np.percentile(qpp_scores, 66)
mask_low  = qpp_scores < t1
mask_mid  = (qpp_scores >= t1) & (qpp_scores < t2)
mask_high = qpp_scores >= t2

def make_rouge(qpp, gain_low=0.057, gain_mid=0.012, gain_high=0.002):
    """Baseline ROUGE-L + QPP-guided gain per tertile."""
    base_rouge = 0.20 + 0.25 * qpp + np.random.randn(len(qpp)) * 0.05
    base_rouge = np.clip(base_rouge, 0.05, 0.80)
    gain = np.where(qpp < t1, gain_low,
           np.where(qpp < t2, gain_mid, gain_high))
    guided_rouge = np.clip(base_rouge + gain + np.random.randn(len(qpp))*0.01, 0, 1)
    return base_rouge, guided_rouge

base_rouge, qpp_rouge = make_rouge(qpp_scores)

# BERTScore
base_bert  = np.clip(0.40 + 0.20 * qpp_scores + np.random.randn(N)*0.04, 0.2, 0.9)
qpp_bert   = np.clip(base_bert + 0.022 * (1.5 - qpp_scores), 0.2, 0.9)

# Token-F1
base_f1    = np.clip(0.35 + 0.20 * qpp_scores + np.random.randn(N)*0.05, 0.1, 0.85)
qpp_f1     = np.clip(base_f1 + 0.023 * (1.2 - qpp_scores), 0.1, 0.85)

# Length-heuristic baseline
heur_rouge = np.clip(base_rouge + 0.008 + np.random.randn(N)*0.01, 0, 1)

print(f'Simulated {N} queries.')
print(f'Low-QPP (hard):   {mask_low.sum():3d} queries (k→50)')
print(f'Medium-QPP:       {mask_mid.sum():3d} queries (k→30)')
print(f'High-QPP (easy):  {mask_high.sum():3d} queries (k→20)')

## 3. Table 11 — Overall Generation Quality Comparison

In [ ]:
# Paper values (Table 11)
TABLE11_PAPER = pd.DataFrame({
    'Method':    ['Baseline RAG', 'QPP-Guided RAG'],
    'ROUGE-L':   [0.360, 0.383],
    'BERTScore': [0.437, 0.459],
    'F1':        [0.400, 0.423],
}).set_index('Method')

# Gains
TABLE11_PAPER.loc['Δ (Gain)'] = TABLE11_PAPER.loc['QPP-Guided RAG'] - TABLE11_PAPER.loc['Baseline RAG']

print('=== Table 11: Baseline RAG vs QPP-Guided RAG ===')
display(TABLE11_PAPER.style
    .format('{:.3f}')
    .apply(lambda col: [
        '', '',
        'background-color: #c8e6c9; font-weight: bold'
    ], axis=0)
    .set_caption('Table 11: QPP-guided RAG outperforms baseline across all metrics.')
)

print('\n--- Computed from synthetic data ---')
computed = pd.DataFrame({
    'Method':    ['Baseline RAG', 'QPP-Guided RAG', 'Δ (Gain)'],
    'ROUGE-L':   [base_rouge.mean(), qpp_rouge.mean(), qpp_rouge.mean() - base_rouge.mean()],
    'BERTScore': [base_bert.mean(),  qpp_bert.mean(),  qpp_bert.mean()  - base_bert.mean()],
    'F1':        [base_f1.mean(),    qpp_f1.mean(),    qpp_f1.mean()    - base_f1.mean()],
}).set_index('Method')
display(computed.style.format('{:.4f}'))

## 4. Table 12 — Per QPP-Group Generation Gains

In [ ]:
# Paper values (Table 12)
TABLE12_PAPER = pd.DataFrame({
    'QPP Group':   ['Low-QPP', 'Medium-QPP', 'High-QPP'],
    'Baseline':    [0.282, 0.358, 0.433],
    'QPP-Guided':  [0.339, 0.370, 0.435],
    'Gain':        [+0.057, +0.012, +0.002],
}).set_index('QPP Group')

print('=== Table 12: ROUGE-L Gains by QPP Difficulty Group ===')
display(TABLE12_PAPER.style
    .format({'Baseline': '{:.3f}', 'QPP-Guided': '{:.3f}', 'Gain': '{:+.3f}'})
    .bar(subset=['Gain'], color=['#ef9a9a', '#a5d6a7'], align='zero')
    .set_caption('Table 12: Largest gains for Low-QPP (hard) queries. '
                 'Confirms QPP-guided adaptation targets failure-prone queries.')
)

# Computed from synthetic data
print('\n--- Computed from synthetic data ---')
for label, mask in [('Low-QPP', mask_low), ('Medium-QPP', mask_mid), ('High-QPP', mask_high)]:
    b = base_rouge[mask].mean()
    g = qpp_rouge[mask].mean()
    print(f'{label:15s}:  Baseline={b:.4f}  QPP-Guided={g:.4f}  Δ={g-b:+.4f}')

## 5. Figure 6 — Proof-of-Concept Example Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: Per-query ROUGE-L scatter (baseline vs QPP-guided) ─────────────────
ax = axes[0]
scatter_colors = np.where(mask_low, '#E53935', np.where(mask_mid, '#FB8C00', '#43A047'))
ax.scatter(base_rouge, qpp_rouge, c=scatter_colors, alpha=0.5, s=15, edgecolors='none')
lim = [0.0, 0.85]
ax.plot(lim, lim, 'k--', lw=1, alpha=0.5, label='y = x (no change)')
ax.set_xlabel('Baseline ROUGE-L')
ax.set_ylabel('QPP-Guided ROUGE-L')
ax.set_title('Per-Query ROUGE-L: Baseline vs QPP-Guided\n(points above diagonal = improvement)')
legend_patches = [
    mpatches.Patch(color='#E53935', label=f'Low-QPP (k→50, n={mask_low.sum()})'),
    mpatches.Patch(color='#FB8C00', label=f'Medium-QPP (k→30, n={mask_mid.sum()})'),
    mpatches.Patch(color='#43A047', label=f'High-QPP (k→20, n={mask_high.sum()})'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='upper left')
ax.set_xlim(lim); ax.set_ylim(lim)

# ── Right: Bar chart — gains per difficulty group ────────────────────────────
ax2 = axes[1]
groups = ['Low-QPP\n(k→50)', 'Medium-QPP\n(k→30)', 'High-QPP\n(k→20)']
baseline_vals  = [0.282, 0.358, 0.433]    # Table 12 paper values
qpp_vals       = [0.339, 0.370, 0.435]
heuristic_vals = [0.290, 0.362, 0.434]    # Section 6 heuristic

x  = np.arange(len(groups))
w  = 0.25
ax2.bar(x - w, baseline_vals,  w, label='Baseline RAG', color='#90A4AE', edgecolor='white')
ax2.bar(x,     qpp_vals,       w, label='QPP-Guided RAG', color='#1E88E5', edgecolor='white')
ax2.bar(x + w, heuristic_vals, w, label='Length Heuristic', color='#FFB300', edgecolor='white')

# Annotate gains
for i, (b, q, label) in enumerate(zip(baseline_vals, qpp_vals, groups)):
    gain = q - b
    ax2.text(i, q + 0.005, f'+{gain:.3f}', ha='center', va='bottom',
             fontsize=9, color='#1E88E5', fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels(groups)
ax2.set_ylabel('ROUGE-L')
ax2.set_title('ROUGE-L Gain by QPP Group\n(Table 12 + Section 6 heuristic comparison)')
ax2.legend(fontsize=9)
ax2.set_ylim(0.20, 0.50)

plt.suptitle('RQ5: Practical Utility of QPP in RAG', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/rq5_adaptive_rag_gains.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → outputs/figures/rq5_adaptive_rag_gains.png')

## 6. Heuristic Comparison (Section 6)

In [ ]:
# Δ ROUGE-L: QPP-guided vs length heuristic vs baseline (paper values)
comparison_df = pd.DataFrame({
    'Method':   ['Baseline RAG', 'Length Heuristic (no learning)', 'QPP-Guided RAG'],
    'ROUGE-L':  [0.360, 0.368, 0.383],
    'BERTScore':[0.437, 0.442, 0.459],
    'ROUGE-L Δ':[0.000, +0.008, +0.023],
    'BERTScore Δ':[0.000, +0.005, +0.022],
}).set_index('Method')

print('=== Section 6: QPP-Guided vs Length-Heuristic vs Baseline ===')
print('Key finding: QPP-guided ROUGE-L gain (+0.023) > heuristic gain (+0.008)')
print('This confirms the benefit comes from the LEARNED retrieval-quality signal,')
print('not merely from varying k.\n')

display(comparison_df.style
    .format({
        'ROUGE-L': '{:.3f}', 'BERTScore': '{:.3f}',
        'ROUGE-L Δ': '{:+.3f}', 'BERTScore Δ': '{:+.3f}'
    })
    .highlight_max(subset=['ROUGE-L Δ', 'BERTScore Δ'], color='#c8e6c9')
    .set_caption('QPP-guided gains (Δ=+0.023) substantially exceed '
                 'the learning-free heuristic (Δ=+0.008)')
)

## 7. Optional: Full End-to-End Run

Uncomment and run this cell if you have:
- GPU with ≥16GB VRAM
- MS MARCO Document dataset downloaded
- Dense FAISS index built
- A HuggingFace token (for LLaMA)

In [ ]:
# ─── UNCOMMENT TO RUN END-TO-END ──────────────────────────────────────────────
# import subprocess, sys
#
# # 1. Train QPP model + save
# subprocess.run([
#     sys.executable, '../run_pipeline.py',
#     '--dataset',   'msmarco_document',
#     '--retriever', 'dense',
#     '--model',     'random_forest',
#     '--mode',      'qpp_only',
#     '--max_queries', '1000',
#     '--save_model', '../outputs/checkpoints/rf_msmarco_doc.pkl',
# ], check=True)
#
# # 2. Run QPP-guided adaptive RAG with BART-large
# subprocess.run([
#     sys.executable, '../run_pipeline.py',
#     '--dataset',   'msmarco_document',
#     '--retriever', 'dense',
#     '--generator', 'bart',
#     '--model',     'random_forest',
#     '--mode',      'adaptive_rag',
#     '--max_queries', '1000',
#     '--model_path', '../outputs/checkpoints/rf_msmarco_doc.pkl',
#     '--output_dir', '../outputs/predictions/',
# ], check=True)
#
# print('End-to-end run complete. Results in ../outputs/predictions/')
print('(Cell is commented out — see instructions above to run the full pipeline.)')

## 8. Takeaways

| Finding | Evidence |
|---|---|
| QPP-guided RAG improves ROUGE-L by **+0.023** overall | Table 11 |
| Largest gains for **Low-QPP** queries: ROUGE-L +0.057 | Table 12 |
| Learning-free heuristic gives only +0.008 (3× smaller) | Section 6 |
| Gain pattern: large for hard queries, negligible for easy ones | Table 12 |
| Adaptive depth: k=50 (hard), k=30 (medium), k=20 (easy) | Eq. 46 |

> **Core message (RQ5):** QPP functions as a practical, lightweight, and
> architecture-agnostic diagnostic layer that makes retrieval behaviour
> **visible and actionable** without requiring any modification to the
> underlying retrieval or generation components.